In [1]:
import os
import pandas as pd
import pyarrow.parquet as pq

In [2]:
base_path = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line"
output_dir = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered"

measurements_file = os.path.join(base_path, "measurements_single_line.parquet")
filtered_output_file = os.path.join(output_dir, "filtered_measurements.parquet")
final_output_file = os.path.join(output_dir, "filtered_measurements_encoded.parquet")

## Step 1: Filter columns with PyArrow

In [ ]:
# Columns you want to keep
keep_columns = [
    "measure_step_number", "measure_value", "created_at", "booking_id",
    "book_state", "serial_number_id",
    "station_id", "measurement_name", "measurement_unit",
    "lower_limit", "upper_limit"
]

In [ ]:
# Read the Parquet file into a PyArrow Table
table = pq.read_table(measurements_file)

In [ ]:
# Drop columns not in keep list
columns_to_drop = [col for col in table.column_names if col not in keep_columns]
filtered_table = table.drop(columns_to_drop)

In [ ]:
# Write filtered table back to Parquet
pq.write_table(filtered_table, final_output_file)
print(f"Reduced Parquet saved to: {final_output_file}")

## Step 2: Encode the measurement_name and measurement_unit columns, Merge upper and lower limit

In [3]:
# Load the reduced dataset for encoding
df = pd.read_parquet(filtered_output_file)

In [4]:
# Convert 'lower_limit', 'upper_limit' and 'measure_value' to numeric (handle strings/mixed types)
numeric_cols = ['measure_value', 'lower_limit', 'upper_limit']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [5]:
# Drop rows where measure_value or limits are missing
df = df.dropna(subset=['measure_value', 'lower_limit', 'upper_limit'])

In [6]:
# Frequency Encoding for 'measurement_name'
measurement_name_freq = df['measurement_name'].value_counts()
df['measurement_name_encoded'] = df['measurement_name'].map(measurement_name_freq)

In [7]:
# Handle missing for 'measurement_unit'
df['measurement_unit'] = df['measurement_unit'].fillna('missing')

In [8]:
# Frequency Encoding for 'measurement_unit'
measurement_unit_freq = df['measurement_unit'].value_counts()
df['measurement_unit_encoded'] = df['measurement_unit'].map(measurement_unit_freq)

In [9]:
# Drop original categorical columns to save space
df.drop(columns=['measurement_name', 'measurement_unit'], inplace=True)

In [10]:
# Create 'is_within_limits' column to merge lower_limit and upper_limit
df['is_within_limits'] = ((df['measure_value'] >= df['lower_limit']) &
                          (df['measure_value'] <= df['upper_limit'])).astype(int)

In [11]:
# Save final encoded dataset
df.to_parquet(final_output_file, index=False)
print(f"Final encoded Parquet saved to: {final_output_file}")

Final encoded Parquet saved to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered\filtered_measurements_encoded.parquet


## Analyse columns

In [3]:
filtered_measurements_file = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered\filtered_measurements_encoded.parquet"

In [4]:
# Load filtered measurements for analysis
df = pd.read_parquet(filtered_measurements_file)

In [5]:
df.isnull().mean().sort_values(ascending=False)

measure_value               0.037813
measure_step_number         0.000000
created_at                  0.000000
booking_id                  0.000000
book_state                  0.000000
part_number                 0.000000
serial_number_id            0.000000
station_id                  0.000000
lower_limit                 0.000000
upper_limit                 0.000000
measurement_name_encoded    0.000000
measurement_unit_encoded    0.000000
is_within_limits            0.000000
dtype: float64

In [7]:
# Number of unique part_number values
unique_part_numbers = df['part_number'].nunique()
print(f"Number of unique part_number values: {unique_part_numbers}")

Number of unique part_number values: 7


In [8]:
# Top 10 most frequent part_number values
top_part_numbers = df['part_number'].value_counts().head(10)
print("\nTop 10 most frequent part_number values:\n")
print(top_part_numbers)


Top 10 most frequent part_number values:

part_number
5a6867de    38044628
9d9d8f8a     4394655
8852ee9e     3838895
a13143e7     2751440
b4044eb8      299136
0cfa8302      271749
793c135e        6156
Name: count, dtype: int64


In [9]:
# Percentage of missing part_number values
missing_percentage = df['part_number'].isnull().mean() * 100
print(f"\nPercentage of missing values in 'part_number': {missing_percentage:.2f}%")


Percentage of missing values in 'part_number': 0.00%


In [10]:
# Analyze cardinality and sparsity for 'measurement_name' and 'measurement_unit'
analysis_columns = ['measurement_name', 'measurement_unit']

for col in analysis_columns:
    print(f"\n=== {col} ===")
    print(f"Unique Values: {df[col].nunique()}")
    print(f"% Missing: {df[col].isna().mean() * 100:.2f}%")

    print("\nTop 10 frequent values:")
    print(df[col].value_counts().head(10))


=== measurement_name ===
Unique Values: 1407
% Missing: 0.00%

Top 10 frequent values:
measurement_name
b8b01a5d    81533
9f78715d    81533
dd0683c4    81533
9aecc867    81533
b140163b    81533
b278ea12    81533
220c497e    81533
2de76d79    81533
2febca25    81533
9c79f21c    81533
Name: count, dtype: int64

=== measurement_unit ===
Unique Values: 25
% Missing: 6.07%

Top 10 frequent values:
measurement_unit
mV      13236260
kOhm     5273521
V        5103493
nF       4531304
KOhm     3775669
 Ohm     2581331
O        2434361
x        1956792
%        1629900
         1174783
Name: count, dtype: int64


In [11]:
# Show overall value counts for understanding cardinality distribution
print("\n=== Summary of Unique Values ===")
summary = df[analysis_columns].nunique().rename("Unique Count")
print(summary)


=== Summary of Unique Values ===
measurement_name    1407
measurement_unit      25
Name: Unique Count, dtype: int64
